# RAG for Critical Thinking Churn Analysis
**Google Colab + Gemini 3.8 Flash + Embeddings**

This notebook shows how Retrieval-Augmented Generation supports critical thinking.

Instead of asking an LLM to invent reasons for the churn spike, we:
1. Retrieve only relevant internal documents
2. Force the model to answer from that evidence
3. Explicitly look for what the evidence does *not* support

This matches the critical-thinking principle: evaluate a topic from all sides using available evidence rather than a single preferred story.

In [ ]:
!pip install -q google-generativeai sentence-transformers pandas numpy scikit-learn

In [ ]:
import google.generativeai as genai
from getpass import getpass
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

api_key = getpass('Enter Gemini API key: ')
genai.configure(api_key=api_key)
llm = genai.GenerativeModel('gemini-3.8-flash')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('Ready')

## Internal Knowledge Base

In [ ]:
documents = [
    {'title': 'UI Redesign', 'content': 'Major UI redesign 6 weeks ago. Users reported difficulty finding features. Navigation tickets increased.'},
    {'title': 'Support Themes', 'content': 'Confusion after UI update and usage guidance requests. Low-usage accounts generate more tickets.'},
    {'title': 'Past Churn Analysis', 'content': 'Usage below 10 hours in first 60 days strongly predicted 90-day churn.'},
    {'title': 'Pricing Note', 'content': 'Price increase hit ~60% of customers, mainly Basic and Pro.'},
    {'title': 'Onboarding', 'content': 'New users drop off before first key action. In-app prompts recommended.'}
]
texts = [d['title'] + '. ' + d['content'] for d in documents]
embeddings = embedder.encode(texts)
print('Embeddings shape:', embeddings.shape)

## Retrieval + Grounded Generation

In [ ]:
def retrieve(query, docs, embs, k=3):
    q = embedder.encode([query])
    sims = cosine_similarity(q, embs)[0]
    idx = np.argsort(sims)[::-1][:k]
    return [(docs[i], float(sims[i])) for i in idx]

def rag(question):
    retrieved = retrieve(question, documents, embeddings)
    ctx = '\n'.join([f"{d['title']}: {d['content']}" for d,_ in retrieved])
    prompt = (
        'Use only these documents to answer. '
        'If the documents do not support a claim, say so clearly.\n\n'
        f'Documents:\n{ctx}\n\nQuestion: {question}'
    )
    return llm.generate_content(prompt).text, retrieved

## Critical Thinking Examples

In [ ]:
questions = [
    'What evidence, if any, directly links the UI redesign to higher churn?',
    'Generate three alternative hypotheses for the churn increase besides the price increase. Rank them by strength of support in the documents.',
    'What does the evidence not allow us to conclude?',
    'Write a short non-technical summary for leadership that states the strongest signals and the remaining uncertainties.',
    'Given the limited evidence and a tight deadline, what is the most practical next step leadership should take right now? Justify using only the documents.'
]

for q in questions:
    print('=' * 60)
    print('QUESTION:', q)
    print('-' * 60)
    answer, used = rag(q)
    print(answer)
    print('\nRetrieved:', [d['title'] for d, _ in used])
    print()

## How this supports critical thinking
- Avoids single-story bias
- Demands evidence
- Surfaces uncertainty
- Ends with a practical recommendation under constraints